<a href="https://colab.research.google.com/github/shin-noda/leetcode-problemset-97/blob/main/Problem1044_Improved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
class Solution:
    def longestDupSubstring(self, s):
        # Convert string to integers (plus a 0 terminator for the '$' concept)
        # This makes it easy to handle recursion with renamed alphabet characters.
        A = [ord(ch) - ord('a') + 1 for ch in s] + [0]

        # Suffix Array:
        # It's an array of starting indices, sorted according to the lexicographic order
        # of the suffixes starting at those indices.
        # A suffix array is really a permutation of the indices 0...n - 1.
        # Nothing is created or destroyed.
        #
        # Run SA-IS to get the full Suffix Array
        # Note: # We don't hardcode 27.
        # alphabet_size is simply the number of distinct integer symbols
        # in the current string. During recursion, the alphabet becomes
        # the set of summary labels (0,1,2,...), not necessarily letters.
        suffix_array = self.sais(A, max(A) + 1)

        # The first element of suffix_array is always the virtual terminator we added
        # so we slice it out to match the original string length
        suffix_array = suffix_array[1:]

        # Build Longest Common Prefix (LCP) array using Kasai's Algorithm (O(N))
        n = len(s)
        lcp = [0] * n
        rank = [0] * n
        for i, sa_val in enumerate(suffix_array):
            rank[sa_val] = i

        h = 0
        max_len = 0
        start_idx = 0

        for i in range(n):
            if rank[i] > 0:
                j = suffix_array[rank[i] - 1]
                while i + h < n and j + h < n and s[i + h] == s[j + h]:
                    h += 1

            lcp[rank[i]] = h

            if h > max_len:
                max_len = h
                start_idx = i

            if h > 0:
                h -= 1

        if max_len <= 0:
            return ""

        return s[start_idx : start_idx + max_len]


    # Note: Suffix Array Induced Sorting (SA-IS)
    def sais(self, A, alphabet_size):
        n = len(A)

        if n == 0:
            return []

        if n == 1:
            return [0]

        if n == 2:
            if A[0] > A[1]:
                return [1, 0]

            else:
                return [0, 1]

        # Classify types: True for S-type, False for L-type
        # Note: S-type (Small) and L-type (Large)
        t = [False] * n

        # Sentinel terminal is always S-type
        t[n - 1] = True

        # We process backwards because we need to know t[i + 1] when
        # we process t[i] and the only value we know initially is
        # t[n - 1] which is the last value.
        # If we start from i = 0, we don't know what t[1] is.
        for i in range(n - 2, -1, -1):
            if A[i] < A[i + 1]:
                # S-type: suffix(i) is lexicographically smaller than suffix(i+1)
                t[i] = True

            elif A[i] > A[i + 1]:
                # Note: L-Type: it's lexicographically larger than or equal to
                # its right neighbour, suffix (i + 1)
                t[i] = False

            else:
                t[i] = t[i + 1]


        # Check if an indexis Leftmost S-type (LMS)
        def is_lms(i):
            # LMS position: it's an index where the type changes from L to S.
            # Note: t[i] = True represents an S-Type and False for an L-Type.
            # So we look for i's where L -> S and t[i] = True which is an S-Type.
            return i > 0 and t[i] and not t[i - 1]


        # Compute bucket sizes and boundaries
        buckets = [0] * alphabet_size
        for x in A:
            buckets[x] += 1


        def get_buckets():
            # head[i]: Start position of bucket i
            head = [0] * alphabet_size

            # tail[i]: End position (exclusive) of bucket i
            tail = [0] * alphabet_size
            acc = 0
            for i, count in enumerate(buckets):
                head[i] = acc
                acc += count
                tail[i] = acc

            return head, tail


        # The induced Sorting core pass
        def induce(lms_indices):
            sa = [-1] * n

            # Note: in induce(), we call get_buckets() three times
            # because we need fresh 'head' and 'tail' for each time.
            head, tail = get_buckets()

            # Place LMS elements at the tails of their buckets
            # # Buckets are filled from the tail (right to left), so we process
            # LMS positions in reverse to match that insertion direction.
            # This preserves the invariants required by induced sorting.
            # We have to process LMS first. Otherwise, sa[i] > 0 won't be true later.
            for idx in reversed(lms_indices):
                c = A[idx]

                # tail[c] points just past thee last valid slot,
                # so that's why we do -1.
                tail[c] -= 1

                # Place suffix starting at idx into the last available slot of its bucket
                sa[tail[c]] = idx

            # Induce L-type elements from left to right
            # We induce new sorted positions from already sorted ones
            head, tail = get_buckets()
            for i in range(n):
                # sa[i] > 0 means this slot already contains a valid suffix index.
                # (Index 0 is skipped because there is no suffix starting at -1.)
                if sa[i] > 0 and not t[sa[i] - 1]:
                    c = A[sa[i] - 1]

                    # The index is head[c] because L-type suffixes
                    # belong at the front of buckets
                    sa[head[c]] = sa[i] - 1
                    head[c] += 1

            # Induce S-type elements from right to left
            head, tail = get_buckets()
            for i in range(n - 1, -1, -1):
                if sa[i] > 0 and t[sa[i] - 1]:
                    c = A[sa[i] - 1]
                    tail[c] -= 1
                    sa[tail[c]] = sa[i] - 1

            return sa


        # Collect all raw LMS positions
        lms_positions = [i for i in range(n) if is_lms(i)]

        # Initial guess of LMS sorting to see if there are naming duplicates
        guessed_sa = induce(lms_positions)

        # Filter guessed_sa to find sorted LMS suffixes
        sorted_lms = [idx for idx in guessed_sa if is_lms(idx)]

        # Assign integer names to LMS substrings.
        # Equal LMS substrings receive the same name.
        lms_names = [-1] * n
        name_counter = 0
        if sorted_lms:
            lms_names[sorted_lms[0]] = name_counter

        for i in range(1, len(sorted_lms)):
            u, v = sorted_lms[i - 1], sorted_lms[i]
            is_equal = True
            for d in range(n):
                # Check if LMS substrings differ or if we hit the next LMS boundaries
                if A[u + d] != A[v + d] or t[u + d] != t[v + d]:
                    is_equal = False
                    break

                if d > 0 and (is_lms(u + d) or is_lms(v + d)):
                    break

            if not is_equal:
                name_counter += 1

            lms_names[v] = name_counter

        # Form a summary string out of the labels
        summary_A = [lms_names[i] for i in lms_positions]
        summary_alphabet_size = name_counter + 1

        # Recursive Call Rule:
        # If every LMS block got a unique name, sorting is complete.
        # Otherwise, recurse to sort the summary string.
        if summary_alphabet_size < len(summary_A):
            # The purpose of the recursion is to determine the correct order of the LMS substrings
            # when some of them are indistinguishable
            summary_sa = self.sais(summary_A, summary_alphabet_size)

        else:
            summary_sa = [-1] * len(summary_A)
            for i, name in enumerate(summary_A):
                summary_sa[name] = i

        # map back to find the exact, absolute
        ordered_lms = [lms_positions[summary_sa[i]] for i in range(len(summary_A))]

        return induce(ordered_lms)

In [11]:
solution = Solution()

In [12]:
s = "banana"

print(solution.longestDupSubstring(s))

ana


In [13]:
s = "abcd"

print(solution.longestDupSubstring(s))